# SE446 — Week 9B: Spark MLlib Lab
## Build an End-to-End ML Pipeline for Arrest Prediction

**Course:** SE446 — Big Data Systems  
**Week:** 09 | Session 9B (Wednesday)  

---

### This notebook runs in THREE environments:

| Mode | Where | Data | How |
|------|-------|------|-----|
| **Local** | Your laptop | 10,000 generated rows | `pip install pyspark`, run in Jupyter |
| **Google Colab** | Browser | 10,000 generated rows | Open notebook, click Run All |
| **Cluster** | SSH to cluster | 7M+ real rows on HDFS | `pyspark --master yarn` |

The notebook **auto-detects** your environment. Same code, same pipeline, different scale.

---

### Learning Objectives

By the end of this lab you will be able to:
1. Build a Spark MLlib `Pipeline` with `StringIndexer`, `VectorAssembler`, and a classifier
2. Train and evaluate Random Forest, Logistic Regression, and GBT models
3. Interpret a confusion matrix and feature importances
4. Compare models and understand when to use each
5. Tune hyperparameters with `CrossValidator`

---
## Part 0: Environment Setup (Auto-Detect)

Run the next two cells first. They install dependencies if needed (Colab), then detect whether you are on the cluster, Colab, or your laptop.

In [ ]:
import os, sys, subprocess

# --- Step 1: Detect environment and install if needed ---
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Google Colab detected — installing PySpark...")
    os.system("pip install -q pyspark")
    print("PySpark installed.")

# --- Step 2: Auto-detect cluster vs local ---
def detect_environment():
    """Detect if we are on the Hadoop cluster, Colab, or a local machine."""
    if IN_COLAB:
        return "colab"
    try:
        result = subprocess.run(
            ["hdfs", "dfs", "-test", "-e", "/data/chicago_crimes.csv"],
            capture_output=True, timeout=5
        )
        if result.returncode == 0:
            return "cluster"
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return "local"

ENV = detect_environment()
print(f"\nEnvironment detected: {ENV.upper()}")

if ENV == "cluster":
    print("  -> Using YARN + HDFS (full Chicago Crimes dataset)")
    print("  -> Data: hdfs:///data/chicago_crimes.csv")
elif ENV == "colab":
    print("  -> Running on Google Colab (local mode, generated data)")
    print("  -> PySpark runs inside this notebook — no cluster needed")
else:
    print("  -> Using local mode (generated sample data)")
    print("  -> No cluster needed — running on your laptop")

In [ ]:
from pyspark.sql import SparkSession

if ENV == "cluster":
    spark = SparkSession.builder \
        .appName("SE446_W09B_MLlib_Lab") \
        .master("yarn") \
        .config("spark.sql.shuffle.partitions", "4") \
        .getOrCreate()
else:
    # Works for both local laptop AND Google Colab
    spark = SparkSession.builder \
        .appName("SE446_W09B_MLlib_Lab_Local") \
        .master("local[*]") \
        .config("spark.sql.shuffle.partitions", "4") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"Environment: {ENV}")

---
## Part 1: Load and Explore Data

### Why this step matters

Before building any ML model, you need to **understand your data**. This means:
- How many rows? (determines if you need distributed ML or scikit-learn is enough)
- What columns are available? (which could be useful features?)
- Is the target variable balanced? (if 95% of rows are one class, accuracy is misleading)

**The key question:** Can we predict whether a crime will result in an **arrest** based on the crime type, location, time, and whether it was domestic?

### How the data works in this notebook

| Mode | What happens |
|------|-------------|
| **Cluster** | Loads 7M+ real Chicago crime records from HDFS |
| **Local/Colab** | Generates 10,000 realistic rows in memory using known arrest rate patterns (e.g., NARCOTICS = 85% arrest rate, THEFT = 10%) |

The generated data mimics real patterns so you learn the same concepts regardless of environment.

In [ ]:
from pyspark.sql.functions import col, avg, count, hour, to_timestamp

if ENV == "cluster":
    # ---- CLUSTER MODE: Load real data from HDFS ----
    raw_df = spark.read.csv(
        "hdfs:///data/chicago_crimes.csv",
        header=True, inferSchema=True
    )
    # Extract Hour from Date and select relevant columns
    df = raw_df.withColumn(
        "Hour", hour(to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a"))
    )
    df = df.select(
        col("District"),
        col("Primary Type").alias("PrimaryType"),
        col("Hour"),
        col("Domestic").cast("string").alias("Domestic_str"),
        col("Arrest")
    ).dropna()
    df = df.withColumn("label", col("Arrest").cast("integer"))

else:
    # ---- LOCAL MODE: Generate realistic sample data ----
    from pyspark.sql import Row
    import random
    random.seed(42)

    crime_profiles = {
        "NARCOTICS":         0.85,
        "PROSTITUTION":      0.80,
        "WEAPONS VIOLATION": 0.60,
        "BATTERY":           0.30,
        "ASSAULT":           0.25,
        "ROBBERY":           0.15,
        "THEFT":             0.10,
        "BURGLARY":          0.08,
        "MOTOR VEHICLE THEFT": 0.06,
        "CRIMINAL DAMAGE":   0.05,
    }
    districts = list(range(1, 26))

    def generate_row():
        crime_type = random.choice(list(crime_profiles.keys()))
        base_rate = crime_profiles[crime_type]
        district = random.choice(districts)
        hour_val = random.randint(0, 23)
        domestic = random.random() < 0.15
        arrest_prob = base_rate + (0.20 if domestic else 0)
        if 2 <= hour_val <= 5:
            arrest_prob -= 0.10
        arrest_prob = max(0.01, min(0.99, arrest_prob))
        arrest = random.random() < arrest_prob
        return Row(
            District=district, PrimaryType=crime_type,
            Hour=hour_val, Domestic_str=str(domestic).lower(),
            Arrest=arrest, label=int(arrest)
        )

    rows = [generate_row() for _ in range(10000)]
    df = spark.createDataFrame(rows)

print(f"Dataset: {df.count():,} rows")
df.printSchema()
df.show(5)

In [ ]:
# --- Class distribution ---
print("=== Arrest Distribution ===")
total = df.count()
df.groupBy("label").count() \
    .withColumn("percentage", (col("count") / total * 100).cast("decimal(5,2)")) \
    .orderBy("label").show()

In [ ]:
# --- Arrest rate by crime type ---
print("=== Arrest Rate by Crime Type ===")
df.groupBy("PrimaryType") \
    .agg(
        count("*").alias("total"),
        avg(col("label")).alias("arrest_rate")
    ) \
    .orderBy(col("arrest_rate").desc()) \
    .show(15)

### Questions — Part 1

1. What is the overall arrest rate? Is this dataset balanced or imbalanced?
2. Which crime type has the highest arrest rate? Does this match your intuition?
3. **(Cluster only)** How many rows are in the full dataset? How long did loading take?

*Your answers here:*

1. 
2. 
3. 

---
## Part 2: Feature Engineering

### What is feature engineering?

ML algorithms only understand **numbers**. Our raw data has strings like `"THEFT"` and `"true"` that a model cannot process directly. Feature engineering converts raw data into a numeric format the model can learn from.

### The three transformers we use

| Transformer | What it does | Example | scikit-learn equivalent |
|------------|-------------|---------|------------------------|
| **StringIndexer** | Converts a text column to a number (most frequent = 0, next = 1, ...) | `"THEFT"` -> `0.0`, `"BATTERY"` -> `1.0` | `LabelEncoder` |
| **VectorAssembler** | Combines multiple numeric columns into a single `features` vector | `[District=8, crime_idx=0, Hour=14, domestic=1]` -> `[8.0, 0.0, 14.0, 1.0]` | `X = df[["col1","col2"]]` |
| **Pipeline** | Chains all steps together so they execute in order | Indexer -> Assembler -> Classifier | `sklearn.pipeline.Pipeline` |

### Why a single feature vector?

In scikit-learn, you pass a matrix `X` with multiple columns. In Spark MLlib, all features must be packed into **one column** called `features` containing a `DenseVector` per row. This is because Spark distributes rows across machines -- each worker needs a self-contained feature vector per row.

### Why use a Pipeline?

Without a Pipeline, you risk **data leakage** -- accidentally fitting transformers on test data. The Pipeline guarantees:
- `pipeline.fit(train_df)` -- fits all transformers on **training data only**
- `model.transform(test_df)` -- applies the same learned mappings to test data

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# --- Define transformers ---
crime_indexer = StringIndexer(
    inputCol="PrimaryType",
    outputCol="crime_index",
    handleInvalid="skip"
)

domestic_indexer = StringIndexer(
    inputCol="Domestic_str",
    outputCol="domestic_index",
    handleInvalid="skip"
)

assembler = VectorAssembler(
    inputCols=["District", "crime_index", "Hour", "domestic_index"],
    outputCol="features"
)

print("Transformers defined:")
print("  1. StringIndexer: PrimaryType -> crime_index")
print("  2. StringIndexer: Domestic_str -> domestic_index")
print("  3. VectorAssembler: [District, crime_index, Hour, domestic_index] -> features")

### Inspect what each transformer does

Before building the pipeline, let's **manually apply each transformer** to see how the data changes at each step. This is the most important part of the lab -- understanding what happens inside the pipeline.

**What to look for in the output below:**
- `PrimaryType` (string) becomes `crime_index` (number)
- `Domestic_str` (string) becomes `domestic_index` (number)
- All four numeric columns get packed into one `features` vector

In [ ]:
# --- See the StringIndexer mapping ---
fitted_crime = crime_indexer.fit(df)
print("Crime type -> index mapping (by frequency):")
for i, label in enumerate(fitted_crime.labels[:10]):
    print(f"  {label:25s} -> {float(i)}")
print(f"  ... ({len(fitted_crime.labels)} total types)")

In [ ]:
# --- Trace one row through all stages ---
temp = fitted_crime.transform(df)
temp = domestic_indexer.fit(df).transform(temp)
temp = assembler.transform(temp)

print("=== Data at each pipeline stage (3 sample rows) ===")
temp.select(
    "PrimaryType", "crime_index",
    "District", "Hour",
    "Domestic_str", "domestic_index",
    "features", "label"
).show(3, truncate=False)

In [ ]:
# --- Train/Test Split ---
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
train_df.cache()  # cache — ML algorithms read training data many times

print(f"Training: {train_df.count():,} rows")
print(f"Testing:  {test_df.count():,} rows")

# Check class balance in training set
print("\nTraining set class balance:")
train_df.groupBy("label").count().orderBy("label").show()

### Questions — Part 2

1. Look at the `features` column. What does a vector like `[8.0, 0.0, 14.0, 1.0]` represent? Map each position to its original column.
2. Why must `StringIndexer` run BEFORE `VectorAssembler`?
3. We called `crime_indexer.fit(df)` on the full dataset above for visualization. In a real pipeline, why is this dangerous? (Hint: data leakage)

---
## Part 3: Train a Random Forest

### How Random Forest works (quick refresher)

You already know this from your ML course. Here's the 30-second version:

1. Build **many decision trees** (e.g., 50 or 100), each on a **random subset** of data and features
2. Each tree makes its own prediction independently
3. The final answer is the **majority vote** across all trees

```
Tree 1 (random data subset) --> predicts Arrest
Tree 2 (different subset)   --> predicts No Arrest
Tree 3 (different subset)   --> predicts Arrest
...
Final prediction = majority vote = Arrest (2 vs 1)
```

### What's different in Spark?

In scikit-learn, all trees train on one machine sequentially. In Spark MLlib, **trees are distributed across workers** -- Worker 1 builds trees 1-25, Worker 2 builds trees 26-50, etc. Same algorithm, parallel execution.

### The Pipeline

We chain all steps into one Pipeline: `StringIndexer -> VectorAssembler -> RandomForest`. When you call `pipeline.fit(train_df)`, Spark:
1. Fits the StringIndexers (learns the string-to-number mappings)
2. Applies VectorAssembler (packs features into vectors)
3. Trains the Random Forest (builds all trees)

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
import time

# --- Define classifier ---
NUM_TREES = 100 if ENV == "cluster" else 50  # fewer trees locally for speed

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=NUM_TREES,
    maxDepth=5,
    seed=42
)

# --- Build pipeline ---
pipeline_rf = Pipeline(stages=[
    crime_indexer,
    domestic_indexer,
    assembler,
    rf
])

# --- Train ---
print(f"Training Random Forest ({NUM_TREES} trees)...")
t = time.time()
model_rf = pipeline_rf.fit(train_df)
rf_time = time.time() - t
print(f"Done in {rf_time:.1f}s")

### Understanding the model output

When the model predicts, it adds three new columns to your DataFrame:

| Column | What it contains | scikit-learn equivalent |
|--------|-----------------|------------------------|
| `prediction` | The predicted class: `0.0` (No Arrest) or `1.0` (Arrest) | `model.predict(X)` |
| `probability` | A vector `[P(No Arrest), P(Arrest)]`, e.g., `[0.82, 0.18]` | `model.predict_proba(X)` |
| `rawPrediction` | Raw scores before probability conversion (internal use) | `model.decision_function(X)` |

The model predicts `1.0` (Arrest) when `P(Arrest) > 0.5`.

In [ ]:
# --- Predict ---
predictions_rf = model_rf.transform(test_df)

# See what the model outputs
print("=== Model Output Columns ===")
predictions_rf.select("label", "prediction", "probability").show(5, truncate=False)

### Evaluation metrics -- what each one tells you

| Metric | What it measures | When to use | Danger |
|--------|-----------------|-------------|--------|
| **Accuracy** | % of correct predictions overall | Balanced datasets | Misleading if data is imbalanced |
| **AUC-ROC** | How well the model ranks positives above negatives (0.5 = random, 1.0 = perfect) | Always useful | -- |
| **Precision** | Of all predicted arrests, how many were real? | When false alarms are costly | Ignores missed arrests |
| **Recall** | Of all real arrests, how many did we catch? | When missing a case is costly | Ignores false alarms |
| **F1 Score** | Balance between Precision and Recall | Imbalanced datasets | -- |

**Why not just accuracy?** If 80% of crimes have No Arrest, a model that always says "No Arrest" gets 80% accuracy -- but catches zero criminals. F1 and AUC are better indicators of real model quality.

In [ ]:
# --- Evaluate ---
binary_eval = BinaryClassificationEvaluator(labelCol="label")
mc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction"
)

auc_rf  = binary_eval.evaluate(predictions_rf)
acc_rf  = mc_eval.evaluate(predictions_rf, {mc_eval.metricName: "accuracy"})
f1_rf   = mc_eval.evaluate(predictions_rf, {mc_eval.metricName: "f1"})
prec_rf = mc_eval.evaluate(predictions_rf, {mc_eval.metricName: "weightedPrecision"})
rec_rf  = mc_eval.evaluate(predictions_rf, {mc_eval.metricName: "weightedRecall"})

print("=== Random Forest Metrics ===")
print(f"  AUC-ROC:   {auc_rf:.4f}")
print(f"  Accuracy:  {acc_rf:.4f}")
print(f"  F1 Score:  {f1_rf:.4f}")
print(f"  Precision: {prec_rf:.4f}")
print(f"  Recall:    {rec_rf:.4f}")

### Confusion Matrix -- the full picture

A confusion matrix shows **all four possible outcomes** of a binary prediction:

```
                      Predicted: No Arrest    Predicted: Arrest
Actual: No Arrest         TN (correct)           FP (false alarm)
Actual: Arrest            FN (missed!)           TP (correct)
```

- **TN** (True Negative): Model said No Arrest, and there was no arrest. Good.
- **FP** (False Positive): Model said Arrest, but there was no arrest. Wasted resources.
- **FN** (False Negative): Model said No Arrest, but there WAS an arrest. Missed case -- dangerous!
- **TP** (True Positive): Model said Arrest, and there was an arrest. Good.

**From these four numbers you can compute:**
- Precision = TP / (TP + FP) -- "When I predict arrest, how often am I right?"
- Recall = TP / (TP + FN) -- "Of all real arrests, how many did I catch?"

In [ ]:
# --- Confusion Matrix ---
print("=== Confusion Matrix (Random Forest) ===")
print("label=0: No Arrest | label=1: Arrest")
predictions_rf.groupBy("label", "prediction") \
    .count().orderBy("label", "prediction").show()

In [ ]:
# --- Feature Importances ---
rf_model = model_rf.stages[-1]
feature_names = ["District", "crime_index", "Hour", "domestic_index"]
importances = rf_model.featureImportances.toArray()

print("=== Feature Importances (Random Forest) ===")
for name, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    bar = "#" * int(imp * 40)
    print(f"  {name:<18} {imp:.4f}  {bar}")

### Feature Importances -- which features drive predictions?

Feature importance tells you **how much each feature contributes** to the model's decisions. A higher value means the feature is used more often at key decision points in the trees.

This is the same concept as `model.feature_importances_` in scikit-learn.

---
## Part 4: Train Logistic Regression

### How Logistic Regression works (quick refresher)

Logistic Regression finds a **linear decision boundary** between classes. It computes:

```
P(Arrest) = sigmoid(w0 + w1*District + w2*crime_index + w3*Hour + w4*domestic_index)
```

Where `w1, w2, w3, w4` are learned **weights** (coefficients). A positive weight means the feature **increases** arrest probability; negative means it **decreases** it.

### Why try LR after Random Forest?

| Aspect | Logistic Regression | Random Forest |
|--------|-------------------|---------------|
| Speed | Fast (one pass through data) | Slower (builds 50-100 trees) |
| Interpretability | Coefficients tell you exactly what each feature does | Feature importances are relative, not directional |
| Non-linear patterns | Cannot capture them | Captures them automatically |

**Strategy:** Always train LR first as a **baseline**. If RF is much better, the data has non-linear patterns that LR misses.

### Key parameters

- `maxIter=100`: Maximum gradient descent steps (usually converges in fewer)
- `regParam=0.01`: L2 regularization -- penalizes large coefficients to prevent overfitting. Higher value = simpler model.

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01
)

pipeline_lr = Pipeline(stages=[
    crime_indexer, domestic_indexer, assembler, lr
])

print("Training Logistic Regression...")
t = time.time()
model_lr = pipeline_lr.fit(train_df)
lr_time = time.time() - t
print(f"Done in {lr_time:.1f}s")

In [ ]:
# --- Evaluate ---
predictions_lr = model_lr.transform(test_df)

auc_lr  = binary_eval.evaluate(predictions_lr)
acc_lr  = mc_eval.evaluate(predictions_lr, {mc_eval.metricName: "accuracy"})
f1_lr   = mc_eval.evaluate(predictions_lr, {mc_eval.metricName: "f1"})
prec_lr = mc_eval.evaluate(predictions_lr, {mc_eval.metricName: "weightedPrecision"})
rec_lr  = mc_eval.evaluate(predictions_lr, {mc_eval.metricName: "weightedRecall"})

print("=== Logistic Regression Metrics ===")
print(f"  AUC-ROC:   {auc_lr:.4f}")
print(f"  Accuracy:  {acc_lr:.4f}")
print(f"  F1 Score:  {f1_lr:.4f}")
print(f"  Precision: {prec_lr:.4f}")
print(f"  Recall:    {rec_lr:.4f}")

In [ ]:
# --- Confusion Matrix ---
print("=== Confusion Matrix (Logistic Regression) ===")
predictions_lr.groupBy("label", "prediction") \
    .count().orderBy("label", "prediction").show()

### Understanding LR Coefficients

Unlike Random Forest (which gives relative importances), Logistic Regression gives you **exact coefficients** that tell you the direction and magnitude of each feature's effect:

- **Positive coefficient** (+0.78): This feature **increases** the probability of arrest
- **Negative coefficient** (-0.01): This feature **decreases** the probability of arrest
- **Larger absolute value** = stronger effect

**Important:** LR treats `crime_index` as a number (0, 1, 2, ...), which implies an **order** between crime types. But there's no real order -- THEFT is not "less than" BATTERY. This is why LR often performs poorly with categorical features encoded as indices. Tree-based models don't have this problem because they split on individual values, not ranges.

In [ ]:
# --- LR Coefficients ---
lr_model = model_lr.stages[-1]
coefficients = lr_model.coefficients.toArray()

print("=== Logistic Regression Coefficients ===")
for name, coef in zip(feature_names, coefficients):
    direction = "(+) increases arrest prob" if coef > 0 else "(-) decreases arrest prob"
    print(f"  {name:<18} {coef:+.4f}  {direction}")

### Questions -- Part 4

1. Which coefficient has the largest absolute value? Translate it into a plain-English statement.
2. LR assumes features contribute linearly. Is this reasonable for crime prediction? Give an example of an interaction effect LR cannot capture.
3. What does `regParam=0.01` control? What happens if you set it to 0? To 10?

*Your answers here:*

1. 
2. 
3. 

---
## Part 5: Gradient-Boosted Trees (GBT)

### How GBT differs from Random Forest

Both use decision trees, but in very different ways:

| | Random Forest (Bagging) | GBT (Boosting) |
|---|---|---|
| **How trees are built** | Each tree is independent, trained on a random subset | Trees are built **one after another**, each correcting the errors of the previous |
| **Parallelism** | Easy -- train all trees at once on different workers | Hard -- must wait for Tree 1 before starting Tree 2 |
| **Accuracy** | Good | Often slightly better |
| **Risk** | Low overfitting (averaging reduces variance) | Higher overfitting risk (each tree targets remaining errors) |

```
RF:  Tree1 + Tree2 + Tree3 ... (all independent, averaged)
GBT: Tree1 -> fix errors -> Tree2 -> fix remaining errors -> Tree3 ...
```

You may know XGBoost from your ML course -- Spark's `GBTClassifier` is the same concept.

**Limitation:** Spark's GBT only supports **binary classification** (2 classes). For multiclass, use RF or LR.

In [ ]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=30 if ENV == "cluster" else 20,
    maxDepth=5,
    stepSize=0.1,
    seed=42
)

pipeline_gbt = Pipeline(stages=[
    crime_indexer, domestic_indexer, assembler, gbt
])

print("Training GBT...")
t = time.time()
model_gbt = pipeline_gbt.fit(train_df)
gbt_time = time.time() - t
print(f"Done in {gbt_time:.1f}s")

# Evaluate
predictions_gbt = model_gbt.transform(test_df)
auc_gbt  = binary_eval.evaluate(predictions_gbt)
acc_gbt  = mc_eval.evaluate(predictions_gbt, {mc_eval.metricName: "accuracy"})
f1_gbt   = mc_eval.evaluate(predictions_gbt, {mc_eval.metricName: "f1"})

print(f"\n=== GBT Metrics ===")
print(f"  AUC-ROC:  {auc_gbt:.4f}")
print(f"  Accuracy: {acc_gbt:.4f}")
print(f"  F1 Score: {f1_gbt:.4f}")

---
## Part 6: Model Comparison

In [ ]:
# --- Side-by-side comparison ---
print("=" * 70)
print(f"{'Metric':<20} {'Random Forest':>15} {'Logistic Reg':>15} {'GBT':>15}")
print("=" * 70)
print(f"{'AUC-ROC':<20} {auc_rf:>15.4f} {auc_lr:>15.4f} {auc_gbt:>15.4f}")
print(f"{'Accuracy':<20} {acc_rf:>15.4f} {acc_lr:>15.4f} {acc_gbt:>15.4f}")
print(f"{'F1 Score':<20} {f1_rf:>15.4f} {f1_lr:>15.4f} {f1_gbt:>15.4f}")
print(f"{'Training Time (s)':<20} {rf_time:>15.1f} {lr_time:>15.1f} {gbt_time:>15.1f}")
print("=" * 70)

# Find best by AUC
results = {"Random Forest": auc_rf, "Logistic Regression": auc_lr, "GBT": auc_gbt}
best = max(results, key=results.get)
print(f"\nBest model by AUC-ROC: {best} ({results[best]:.4f})")

### Questions -- Part 6

1. Which model won on AUC? On F1? On training speed? Are the rankings the same across metrics?
2. GBT trains trees sequentially (each corrects the previous). RF trains trees independently. Which parallelizes better on a cluster? Why?
3. If accuracy is very close to the arrest rate (from Part 1), what does that tell you about the model?
4. A police department wants to deploy this model. Which model would you recommend and why? Consider accuracy, speed, and interpretability.

*Your answers here:*

1. 
2. 
3. 
4. 

---
## Part 7: Hyperparameter Tuning with CrossValidator

Instead of manually trying different settings, `CrossValidator` searches automatically — and Spark parallelizes the search across the cluster.

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# --- Define a fresh RF for tuning ---
rf_tune = RandomForestClassifier(
    featuresCol="features", labelCol="label", seed=42
)

pipeline_tune = Pipeline(stages=[
    crime_indexer, domestic_indexer, assembler, rf_tune
])

# --- Parameter grid ---
paramGrid = ParamGridBuilder() \
    .addGrid(rf_tune.numTrees, [20, 50]) \
    .addGrid(rf_tune.maxDepth, [3, 5]) \
    .build()

print(f"Grid: {len(paramGrid)} combinations (2 numTrees x 2 maxDepth)")
print(f"With 3-fold CV: {len(paramGrid) * 3} total models to train")

In [ ]:
# --- Cross-Validate ---
cv = CrossValidator(
    estimator=pipeline_tune,
    estimatorParamMaps=paramGrid,
    evaluator=BinaryClassificationEvaluator(labelCol="label"),
    numFolds=3,
    parallelism=2  # train 2 models in parallel
)

print("Running CrossValidator (this may take a minute)...")
t = time.time()
cvModel = cv.fit(train_df)
cv_time = time.time() - t
print(f"Done in {cv_time:.1f}s")

# --- Results ---
print("\n=== Cross-Validation Results ===")
for params, score in zip(paramGrid, cvModel.avgMetrics):
    trees = params[rf_tune.numTrees]
    depth = params[rf_tune.maxDepth]
    print(f"  numTrees={trees:3d}, maxDepth={depth} -> AUC={score:.4f}")

# Best model
best_auc = binary_eval.evaluate(cvModel.transform(test_df))
print(f"\nBest model AUC on test set: {best_auc:.4f}")

### Questions -- Part 7

1. Which combination of `numTrees` and `maxDepth` performed best?
2. We trained 12 models (4 combos x 3 folds). On a 4-worker cluster with `parallelism=4`, how much faster would this be vs sequential?
3. In scikit-learn, the equivalent is `GridSearchCV`. What is the key advantage of Spark's `CrossValidator` over `GridSearchCV` at scale?

*Your answers here:*

1. 
2. 
3. 

---
## Part 8: Save and Load the Model

### Why save models?

Training a model can take minutes (or hours on large data). Once trained, you want to **reuse it without retraining**:

1. **Train once** on the cluster (expensive)
2. **Save** the trained model to disk (HDFS on cluster, `/tmp/` locally)
3. **Load** it later in a production app (cheap, fast)
4. **Retrain periodically** when new data arrives (weekly/monthly)

### What gets saved?

The entire `PipelineModel` is saved -- not just the classifier. This includes the fitted StringIndexer mappings, the VectorAssembler configuration, and all trained trees. So `loaded_model.transform(new_raw_data)` handles everything end-to-end.

In [ ]:
import tempfile
from pyspark.ml import PipelineModel

if ENV == "cluster":
    save_path = "hdfs:///models/arrest_predictor_rf_v1"
else:
    save_path = os.path.join(tempfile.gettempdir(), "arrest_model_rf")

# Save
model_rf.write().overwrite().save(save_path)
print(f"Model saved to: {save_path}")

# Reload
loaded = PipelineModel.load(save_path)
loaded_preds = loaded.transform(test_df.limit(5))
loaded_preds.select("PrimaryType", "label", "prediction", "probability").show(truncate=False)
print("Model reloaded and working!")

### Questions — Part 8

1. On the cluster, we save to HDFS. On your laptop, we save to `/tmp/`. Why is HDFS better for a distributed environment?
2. The saved model includes fitted StringIndexer mappings. Why is this important? What breaks if you save only the classifier?

*Your answers here:*

1. 
2. 

---
## Cleanup

In [ ]:
spark.stop()
print(f"Lab complete! Environment: {ENV}")
print("SparkSession stopped.")

---
## Reflection Questions

Answer these after completing all parts:

1. The pipeline order is: `StringIndexer -> VectorAssembler -> Classifier`. Could you swap `StringIndexer` and `VectorAssembler`? Why or why not?
2. We used 4 features. Propose **two additional features** you could extract from the existing columns. Justify your choices.
3. A model achieving 80% accuracy sounds good. But if the arrest rate is 20%, a model that always predicts "No Arrest" gets 80% accuracy too. What metric should you use instead, and why?
4. **(Cluster only)** Compare your training times between local (10K rows) and cluster (7M rows). Did training time scale linearly with data size? Why or why not?
5. **(Ethics)** If the model learns that certain districts predict arrests more strongly, how might this reinforce algorithmic bias?

*Your answers here:*

1. 
2. 
3. 
4. 
5. 

---
## Deliverables

Submit this notebook (`.ipynb`) with:
1. All code cells executed with output visible
2. All question cells filled in
3. The 3-model comparison table from Part 6
4. Confusion matrices for RF and LR

---

## Grading

| Component | Points |
|-----------|:------:|
| Part 1–2: Data loading + exploration + feature engineering + 6 questions | 20 |
| Part 3: Random Forest — train, evaluate, importances + 4 questions | 20 |
| Part 4: Logistic Regression + 3 questions | 15 |
| Part 5–6: GBT + model comparison + 4 questions | 15 |
| Part 7: CrossValidator + 3 questions | 15 |
| Part 8: Save/load + 2 questions | 5 |
| Reflection questions (5 questions) | 10 |
| **Total** | **100** |